In [1]:
!head ratings_long.csv

head: cannot open 'ratings_long.csv' for reading: No such file or directory


In [ ]:
r = np.full((20, 1000),fill_value=np.nan)

In [ ]:
df = pd.read_csv('ratings_long.csv')

In [ ]:
for rec in df.itertuples():
    r[rec.userId][rec.movieId] = rec.rating

Note that $r$ matrix is $20 \times 1000$ with only <1\% full (highly sparse)

In [3]:
r

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan,  4., nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

What if we define two matricies
- u = $20 \times 4$
- v = $4 \times 1000$


Then model $r$ as $u \times v$

Problem is we have to learn for $20 \times 4 + 4 \times 1000 = 4080$ parameters (better than 20.000 x 0.99 missing values)

## Problem

1. Define a convex loss function wrt $u$ and $v$
- Solve using gradient descent algorithm explained in **I Do**
- Use any regulatizer $L1$ or $L2$ to prevent overfitting

In [21]:
import numpy as np

obs = np.argwhere(~np.isnan(r))
R   = np.nan_to_num(r)

rng = np.random.default_rng(0)

def make_mask(ij):
    m = np.zeros_like(r, dtype=bool)
    m[ij[:, 0], ij[:, 1]] = True
    return m

# ---- L2-regularize MF egitimi (gradient descent) ----
def train_mf(train_mask, K=4, lr=2.0, lam=0.01, epochs=5000, seed=0):
    rng  = np.random.default_rng(seed)
    u    = rng.normal(0, 0.1, (20,   K))
    v    = rng.normal(0, 0.1, (K, 1000))
    n_tr = train_mask.sum()
    for _ in range(epochs):
        E = train_mask * (R - u @ v)                    # hata SADECE train hucrelerinde
        u -= lr * ((-2.0 / n_tr) * (E   @ v.T) + 2 * lam * u)   # L2
        v -= lr * ((-2.0 / n_tr) * (u.T @ E)   + 2 * lam * v)
    return u, v

def rmse(mask, u, v):
    r_hat = np.clip(u @ v, 1, 5)
    return np.sqrt(((mask * (R - r_hat))**2).sum() / mask.sum())

# ---- 5-Fold Cross-Validation ----
N_FOLDS = 5
fold_id = rng.permutation(len(obs)) % N_FOLDS

print(f"gozlemli={len(obs)}  |  {N_FOLDS}-fold  (~{len(obs)//N_FOLDS} val/fold)\n")
print("5-Fold Cross-Validation adimlari:")
print(f"{'fold':>5} | {'train':>6} | {'val':>5} | {'train_rmse':>10} | {'val_rmse':>9}")
print("-" * 50)

cv_tr, cv_val = [], []
for f in range(N_FOLDS):
    val_ij   = obs[fold_id == f]
    train_ij = obs[fold_id != f]                         # kalan 4 fold -> train
    u, v = train_mf(make_mask(train_ij))
    tr  = rmse(make_mask(train_ij), u, v)
    val = rmse(make_mask(val_ij),   u, v)
    cv_tr.append(tr); cv_val.append(val)
    print(f"{f:5d} | {len(train_ij):6d} | {len(val_ij):5d} | {tr:10.4f} | {val:9.4f}")

print("-" * 50)
print(f"{'ort':>5} | {'':>6} | {'':>5} | {np.mean(cv_tr):10.4f} | {np.mean(cv_val):9.4f}")
print(f"\nGenelleme tahmini = CV val RMSE ortalamasi = {np.mean(cv_val):.4f}  (+/- {np.std(cv_val):.4f})")


gozlemli=200  |  5-fold  (~40 val/fold)

5-Fold Cross-Validation adimlari:
 fold |  train |   val | train_rmse |  val_rmse
--------------------------------------------------
    0 |    160 |    40 |     0.5602 |    2.6263
    1 |    160 |    40 |     0.5499 |    2.2058
    2 |    160 |    40 |     0.5555 |    2.4314
    3 |    160 |    40 |     0.5510 |    2.6247
    4 |    160 |    40 |     0.5576 |    2.1851
--------------------------------------------------
  ort |        |       |     0.5548 |    2.4147

Genelleme tahmini = CV val RMSE ortalamasi = 2.4147  (+/- 0.1926)
